In [3]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter

# Load preprocessed train and test data
X_train_final = pd.read_parquet('../data/cleaned/X_train_preprocessed.parquet')
X_test_final = pd.read_parquet('../data/cleaned/X_test_preprocessed.parquet')
y_train = pd.read_parquet('../data/cleaned/y_train.parquet').squeeze()
y_test = pd.read_parquet('../data/cleaned/y_test.parquet').squeeze()

print("="*70)
print("DATA LOADED SUCCESSFULLY")
print("="*70)
print(f"✓ X_train shape: {X_train_final.shape}")
print(f"✓ X_test shape: {X_test_final.shape}")
print(f"✓ y_train shape: {y_train.shape}")
print(f"✓ y_test shape: {y_test.shape}")

DATA LOADED SUCCESSFULLY
✓ X_train shape: (103986, 40)
✓ X_test shape: (25997, 40)
✓ y_train shape: (103986,)
✓ y_test shape: (25997,)


In [4]:
# ============================================================
# TARGET VARIABLE ANALYSIS
# ============================================================

print("\n" + "="*70)
print("TARGET VARIABLE ANALYSIS")
print("="*70)

train_counts = Counter(y_train)
test_counts = Counter(y_test)

print(f"\nTrain target distribution:")
for label, count in sorted(train_counts.items()):
    pct = (count / len(y_train)) * 100
    print(f"  Class {label}: {count:,} ({pct:.2f}%)")

print(f"\nTest target distribution:")
for label, count in sorted(test_counts.items()):
    pct = (count / len(y_test)) * 100
    print(f"  Class {label}: {count:,} ({pct:.2f}%)")

# Calculate imbalance ratio
minority_class = min(train_counts, key=train_counts.get)
majority_class = max(train_counts, key=train_counts.get)
imbalance_ratio = train_counts[majority_class] / train_counts[minority_class]

print(f"\nImbalance ratio: {imbalance_ratio:.2f}:1")
print(f"Minority class ({minority_class}): {train_counts[minority_class]:,} samples")
print(f"Majority class ({majority_class}): {train_counts[majority_class]:,} samples")


TARGET VARIABLE ANALYSIS

Train target distribution:
  Class 0: 17,178 (16.52%)
  Class 1: 86,808 (83.48%)

Test target distribution:
  Class 0: 4,295 (16.52%)
  Class 1: 21,702 (83.48%)

Imbalance ratio: 5.05:1
Minority class (0): 17,178 samples
Majority class (1): 86,808 samples


In [8]:
X_train_final.dtypes


provider_code             category
year                      category
age_band                  category
gender                     float64
t0_assisted                float64
t0_assisted_by               uint8
t0_symptom_period          float64
t0_previous_surgery        float64
t0_living_arrangements    category
t0_disability              float64
heart_disease                int64
high_bp                      int64
stroke                       int64
circulation                  int64
lung_disease                 int64
diabetes                     int64
kidney_disease               int64
nervous_system               int64
liver_disease                int64
cancer                       int64
depression                   int64
arthritis                    int64
t0_mobility                float64
t0_self_care               float64
t0_activity                float64
t0_discomfort              float64
t0_anxiety                 float64
oks_t0_pain                  uint8
oks_t0_night_pain   

In [11]:
# ============================================================
# ONE-HOT ENCODE CATEGORICAL VARIABLES
# ============================================================

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import joblib

# Define categorical columns to encode
categorical_columns = [
    'provider_code', 'year', 'age_band', 
    't0_living_arrangements'
]

print("="*70)
print("ONE-HOT ENCODING CATEGORICAL VARIABLES")
print("="*70)

# Check which categorical columns exist in the data
existing_categorical = [col for col in categorical_columns if col in X_train_final.columns]
print(f"\nCategorical columns found: {existing_categorical}")

# ============================================================
# FIT ONE-HOT ENCODER ON TRAINING DATA
# ============================================================

# Initialize encoder
encoder = OneHotEncoder(
    drop='first',           # Drop first category to avoid multicollinearity
    sparse_output=False,    # Return dense array (easier to work with)
    handle_unknown='ignore' # Ignore unknown categories in test set
)

# Fit on training data
print(f"\n✓ Fitting encoder on training data...")
encoder.fit(X_train_final[existing_categorical])

# Get feature names for encoded columns
encoded_feature_names = encoder.get_feature_names_out(existing_categorical)
print(f"✓ Encoder fitted - will create {len(encoded_feature_names)} one-hot encoded features")

# ============================================================
# TRANSFORM TRAINING DATA
# ============================================================

print(f"\n✓ Transforming training data...")

# Transform categorical columns
X_train_encoded = encoder.transform(X_train_final[existing_categorical])

# Create DataFrame with encoded features
X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train_final.index
)

# Drop original categorical columns and add encoded ones
X_train_numerical = X_train_final.drop(columns=existing_categorical)
X_train_final_encoded = pd.concat([X_train_numerical, X_train_encoded_df], axis=1)

print(f"✓ Training data shape before encoding: {X_train_final.shape}")
print(f"✓ Training data shape after encoding: {X_train_final_encoded.shape}")

# ============================================================
# TRANSFORM TEST DATA
# ============================================================

print(f"\n✓ Transforming test data...")

# Transform categorical columns (using fitted encoder)
X_test_encoded = encoder.transform(X_test_final[existing_categorical])

# Create DataFrame with encoded features
X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test_final.index
)

# Drop original categorical columns and add encoded ones
X_test_numerical = X_test_final.drop(columns=existing_categorical)
X_test_final_encoded = pd.concat([X_test_numerical, X_test_encoded_df], axis=1)

print(f"✓ Test data shape before encoding: {X_test_final.shape}")
print(f"✓ Test data shape after encoding: {X_test_final_encoded.shape}")

# ============================================================
# VERIFICATION
# ============================================================

print("\n" + "="*70)
print("ENCODING VERIFICATION")
print("="*70)

print(f"\n✓ X_train_final_encoded shape: {X_train_final_encoded.shape}")
print(f"✓ X_test_final_encoded shape: {X_test_final_encoded.shape}")
print(f"✓ Both datasets have same number of features: {X_train_final_encoded.shape[1] == X_test_final_encoded.shape[1]}")

# Check for missing values
print(f"\n✓ Missing values in X_train_final_encoded: {X_train_final_encoded.isna().sum().sum()}")
print(f"✓ Missing values in X_test_final_encoded: {X_test_final_encoded.isna().sum().sum()}")

# Show sample of new columns
print("\n" + "="*70)
print("SAMPLE OF ONE-HOT ENCODED FEATURES")
print("="*70)

# Show first few encoded feature names for each original categorical variable
for original_col in existing_categorical:
    encoded_cols = [col for col in encoded_feature_names if col.startswith(original_col)]
    print(f"\n{original_col} → {len(encoded_cols)} one-hot features:")
    print(f"  Sample: {encoded_cols[:5]}")

# Show data sample
print("\n" + "="*70)
print("DATA PREVIEW (First 5 rows, first 10 columns)")
print("="*70)
print(X_train_final_encoded.iloc[:5, :10])

# ============================================================
# SAVE ENCODED DATA (Optional)
# ============================================================

print("\n" + "="*70)
print("SAVING ENCODED DATA")
print("="*70)

# Save encoded datasets
X_train_final_encoded.to_parquet('../data/cleaned/X_train_encoded.parquet', index=False)
X_test_final_encoded.to_parquet('../data/cleaned/X_test_encoded.parquet', index=False)

# Save encoder for future use
joblib.dump(encoder, '../data/cleaned/onehot_encoder.pkl')

print("✓ X_train_final_encoded saved to '../data/cleaned/X_train_encoded.parquet'")
print("✓ X_test_final_encoded saved to '../data/cleaned/X_test_encoded.parquet'")
print("✓ OneHotEncoder saved to '../data/cleaned/onehot_encoder.pkl'")

# ============================================================
# FEATURE COUNTS BY TYPE
# ============================================================

print("\n" + "="*70)
print("FEATURE BREAKDOWN")
print("="*70)

# Count features by original type
numerical_features = X_train_numerical.columns.tolist()
encoded_features = encoded_feature_names.tolist()

print(f"\nNumerical features (unchanged): {len(numerical_features)}")
print(f"One-hot encoded features (new): {len(encoded_features)}")
print(f"Total features: {len(X_train_final_encoded.columns)}")

print("\n" + "="*70)
print("✓ ONE-HOT ENCODING COMPLETE - DATA READY FOR MODELING")
print("="*70)

# ============================================================
# How to load encoder and apply to new data later:
# ============================================================
"""
# Load the fitted encoder
encoder = joblib.load('../data/processed/onehot_encoder.pkl')

# Apply to new data
X_new_encoded = encoder.transform(X_new[existing_categorical])
X_new_encoded_df = pd.DataFrame(
    X_new_encoded,
    columns=encoder.get_feature_names_out(existing_categorical),
    index=X_new.index
)
X_new_numerical = X_new.drop(columns=existing_categorical)
X_new_final = pd.concat([X_new_numerical, X_new_encoded_df], axis=1)
"""

ONE-HOT ENCODING CATEGORICAL VARIABLES

Categorical columns found: ['provider_code', 'year', 'age_band', 't0_living_arrangements']

✓ Fitting encoder on training data...
✓ Encoder fitted - will create 300 one-hot encoded features

✓ Transforming training data...
✓ Training data shape before encoding: (103986, 40)
✓ Training data shape after encoding: (103986, 336)

✓ Transforming test data...
✓ Test data shape before encoding: (25997, 40)
✓ Test data shape after encoding: (25997, 336)

ENCODING VERIFICATION

✓ X_train_final_encoded shape: (103986, 336)
✓ X_test_final_encoded shape: (25997, 336)
✓ Both datasets have same number of features: True

✓ Missing values in X_train_final_encoded: 0
✓ Missing values in X_test_final_encoded: 0

SAMPLE OF ONE-HOT ENCODED FEATURES

provider_code → 293 one-hot features:
  Sample: ['provider_code_AHH', 'provider_code_AVQ', 'provider_code_NFH01', 'provider_code_NN401', 'provider_code_NN801']

year → 2 one-hot features:
  Sample: ['year_2017/18', 'year

"\n# Load the fitted encoder\nencoder = joblib.load('../data/processed/onehot_encoder.pkl')\n\n# Apply to new data\nX_new_encoded = encoder.transform(X_new[existing_categorical])\nX_new_encoded_df = pd.DataFrame(\n    X_new_encoded,\n    columns=encoder.get_feature_names_out(existing_categorical),\n    index=X_new.index\n)\nX_new_numerical = X_new.drop(columns=existing_categorical)\nX_new_final = pd.concat([X_new_numerical, X_new_encoded_df], axis=1)\n"

📊 Modeling Guide – Where to Start

🎯 1. Problem Summary
- Task: Binary classification
- Target: oks_target_variable (whether patient improvement > 7 points)
- Class Imbalance: ~5:1 ratio (majority class dominates)
- Data: Fully preprocessed, no missing values, ready for modeling

🔧 2. Handling Class Imbalance
- You have several options:
✅ Option A: Class Weights (Recommended first approach)

- Use class_weight='balanced' in your classifier
- No need to modify data
- Works with: LogisticRegression, RandomForest, XGBoost, etc.

✅ Option B: Resampling (SMOTE)

- Apply SMOTE only to training data inside cross-validation
- Use imblearn.pipeline.Pipeline to prevent data leakage
- Best for smaller datasets

✅ Option C: Threshold Tuning

- Train normally, then adjust decision threshold
- Optimize for your preferred metric (F1, recall, precision)


📈 3. Evaluation Metrics
Given class imbalance, DO NOT rely solely on accuracy. Focus on:
Primary Metrics

- F1-Score (balance between precision and recall)
- ROC-AUC (overall discrimination ability)
- Precision-Recall AUC (better for imbalanced data)

Secondary Metrics

- Recall (if you want to catch all positive cases)
- Precision (if false positives are costly)
- Confusion Matrix (understand errors)


🧪 4. Model Selection Strategy
Baseline Models (Start here)

- Logistic Regression (class_weight='balanced')
- Random Forest (class_weight='balanced')
- Dummy Classifier (minimum performance)

Advanced Models (After baseline)

- XGBoost or LightGBM (handle categorical features well)
- CatBoost (native categorical support)
- Ensemble methods (stacking, voting)


🔄 5. Cross-Validation Strategy

- Use StratifiedKFold (5–10 folds)
- Maintains class distribution in each fold
- Report mean ± std of metrics

Important:

- If using SMOTE, apply it inside each CV fold
- Use imblearn.pipeline.Pipeline to prevent leakage


🎛️ 6. Hyperparameter Tuning
- Step 1: Train baseline models with default parameters
- Step 2: Use GridSearchCV or RandomizedSearchCV
- Step 3: Focus on models that performed best in baseline
Key Parameters to Tune
Random Forest
- Plain Textn_estimators: [100, 200, 500]max_depth: [10, 20, None]min_samples_split: [2, 5, 10]class_weight: ['balanced']Show more lines
XGBoost
- Plain Textn_estimators: [100, 200, 500]learning_rate: [0.01, 0.1, 0.3]max_depth: [3, 5, 7]scale_pos_weight: [calculated from class ratio]Show more lines

📊 7. Feature Engineering (Optional)

Feature Interactions:
- Example: age_band * heart_disease
Feature Selection:
- Use feature importance from tree-based models
- Try Recursive Feature Elimination (RFE)
Categorical Encoding:

- One-Hot Encoding for low-cardinality features
- Target Encoding for high-cardinality features (e.g., provider_code)

⚠️ 8. Important Reminders
- ✅ Use stratified cross-validation
- ✅ Focus on F1/ROC-AUC, not just accuracy
- ✅ Handle class imbalance (class weights or SMOTE)
- ✅ Evaluate on test set ONLY ONCE at the end
- ✅ Save your best model with joblib
- ❌ Touch the test set during model development
- ❌ Use SMOTE before train-test split
- ❌ Rely on accuracy alone for imbalanced data
- ❌ Forget to set random_state for reproducibility
- ❌ Oversample test data

# Encoded data

In [ ]:
# Load encoded data

X_train_final_encoded = pd.read_parquet('../data/cleaned/X_train_encoded.parquet')
print("✓ Encoded data loaded successfully")

✓ Encoded data loaded successfully


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib
import numpy as np

# Train and evaluate a logistic regression model (balanced) on encoded data

# Use encoded datasets loaded previously: X_train_final_encoded, X_test_final_encoded, y_train, y_test

# Build pipeline: scaling + logistic regression with class balancing
pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),  # with_mean=False for sparse-like one-hot cols safety
    ("clf", LogisticRegression(class_weight="balanced", solver="saga", max_iter=2000, random_state=42))
])

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# CV scores
roc_scores = cross_val_score(pipe, X_train_final_encoded, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
f1_scores = cross_val_score(pipe, X_train_final_encoded, y_train, cv=cv, scoring="f1", n_jobs=-1)

print(f"CV ROC-AUC: {roc_scores.mean():.4f} ± {roc_scores.std():.4f}")
print(f"CV F1     : {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")

# Fit on full training set
pipe.fit(X_train_final_encoded, y_train)

# Evaluate on test set
y_pred = pipe.predict(X_test_final_encoded)
y_proba = pipe.predict_proba(X_test_final_encoded)[:, 1] if hasattr(pipe.named_steps["clf"], "predict_proba") else None

print("\nTest set classification report:")
print(classification_report(y_test, y_pred, digits=4))

if y_proba is not None:
    print(f"Test ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

print("Confusion matrix (rows: true, cols: pred):")
print(confusion_matrix(y_test, y_pred))

# Save trained pipeline
joblib.dump(pipe, "../data/cleaned/logistic_regression_balanced_pipeline.pkl")
print("\n✓ Saved logistic regression pipeline to '../data/cleaned/logistic_regression_balanced_pipeline.pkl'")